# Análisis de videos en tendencia de YouTube en Gran Bretaña

Proyecto académico del curso Fundamentos de Data Science. Repositorio: `FDS-2026-1-16685`.

## 1. Comprensión del negocio

El objetivo es analizar el comportamiento de los videos en tendencia de YouTube en Gran Bretaña e identificar factores asociados a su rendimiento, usando `views` como variable objetivo.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

DATA_DIR = Path("../data")
CSV_PATH = DATA_DIR / "GBvideos_cc50_202101.csv"
JSON_PATH = DATA_DIR / "GB_category_id.json"
OUTPUT_PATH = DATA_DIR / "GBvideos_limpio_preparado.csv"

pd.set_option("display.max_columns", None)

## 2. Comprensión de los datos

In [ ]:
df = pd.read_csv(CSV_PATH)
df.head()

In [ ]:
print("Filas y columnas:", df.shape)
df.info()

In [ ]:
df.describe()

In [ ]:
pd.DataFrame({
    "nulos": df.isnull().sum(),
    "unicos": df.nunique(),
    "tipo": df.dtypes
})

In [ ]:
print("Duplicados:", df.duplicated().sum())

## 3. Preparación de los datos

In [ ]:
with open(JSON_PATH, "r", encoding="utf-8") as file:
    categorias = json.load(file)

diccionario_categorias = {
    int(item["id"]): item["snippet"]["title"]
    for item in categorias["items"]
}

df["category_name"] = df["category_id"].map(diccionario_categorias).fillna("No clasificada")
df["description"] = df["description"].fillna("Sin descripcion")
df["ratio_likes"] = df["likes"] / (df["dislikes"] + 1)

df.head()

In [ ]:
print("Nulos luego de limpieza:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("Duplicados luego de limpieza:", df.duplicated().sum())

In [ ]:
df.to_csv(OUTPUT_PATH, index=False)
print(f"Dataset limpio guardado en: {OUTPUT_PATH}")

## 4. Análisis exploratorio de datos

In [ ]:
numericas = ["views", "likes", "dislikes", "comment_count", "ratio_likes"]
corr = df[numericas].corr()
corr

In [ ]:
plt.figure(figsize=(8, 5))
plt.imshow(corr)
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
plt.yticks(range(len(corr.index)), corr.index)
plt.colorbar(label="Correlación")
for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center")
plt.title("Correlación entre variables numéricas")
plt.tight_layout()
plt.show()

### Cantidad de videos por categoría

In [ ]:
categoria = df["category_name"].value_counts()
plt.figure(figsize=(10,5))
categoria.plot(kind="bar")
plt.title("Cantidad de videos por categoría")
plt.xlabel("Categoría")
plt.ylabel("Cantidad de videos")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### Visualizaciones por categoría

In [ ]:
categoria_views = df.groupby("category_name")["views"].sum().sort_values(ascending=False)
plt.figure(figsize=(11,5))
categoria_views.plot(kind="bar")
plt.title("Visualizaciones por categoría")
plt.xlabel("Categoría")
plt.ylabel("Total de visualizaciones")
plt.xticks(rotation=45, ha="right")
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()

### Top 10 canales con más visualizaciones

In [ ]:
canales = df.groupby("channel_title")["views"].sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(10,5))
canales.plot(kind="bar")
plt.title("Top 10 canales con más visualizaciones")
plt.xlabel("Canal")
plt.ylabel("Total de visualizaciones")
plt.xticks(rotation=45, ha="right")
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()

### Top 10 videos con más visualizaciones

In [ ]:
top_videos = df.sort_values("views", ascending=False).drop_duplicates(subset="title").head(10)
titulos = [titulo[:35] + "..." if len(titulo) > 35 else titulo for titulo in top_videos["title"]]
plt.figure(figsize=(12,6))
plt.bar(titulos, top_videos["views"])
plt.title("Top 10 videos con más visualizaciones")
plt.xlabel("Video")
plt.ylabel("Visualizaciones")
plt.xticks(rotation=60, ha="right")
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()

### Likes por categoría

In [ ]:
likes_categoria = df.groupby("category_name")["likes"].sum().sort_values(ascending=False)
plt.figure(figsize=(11,5))
likes_categoria.plot(kind="bar")
plt.title("Likes por categoría")
plt.xlabel("Categoría")
plt.ylabel("Cantidad de likes")
plt.xticks(rotation=45, ha="right")
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()

### Promedio de visualizaciones por categoría

In [ ]:
promedio_views = df.groupby("category_name")["views"].mean().sort_values(ascending=False)
plt.figure(figsize=(10,5))
promedio_views.plot(kind="bar")
plt.title("Promedio de visualizaciones por categoría")
plt.xlabel("Categoría")
plt.ylabel("Promedio de visualizaciones")
plt.xticks(rotation=45, ha="right")
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()

### Top 10 canales con más likes

In [ ]:
likes_canales = df.groupby("channel_title")["likes"].sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(11,5))
likes_canales.plot(kind="bar")
plt.title("Top 10 canales con más likes")
plt.xlabel("Canal")
plt.ylabel("Cantidad de likes")
plt.xticks(rotation=45, ha="right")
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()

## 5. Modelado y evaluación

In [ ]:
muestra = df.sample(3000, random_state=42)
plt.figure(figsize=(10,6))
plt.scatter(muestra["likes"], muestra["views"], alpha=0.35, s=15)
plt.title("Gráfico de dispersión entre Likes y Visualizaciones")
plt.xlabel("Cantidad de Likes")
plt.ylabel("Cantidad de Visualizaciones")
plt.gca().xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
features = ["likes", "comment_count", "ratio_likes"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["views"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

modelo = LinearRegression()
modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
mae = mean_absolute_error(y_test, y_pred)

metricas = pd.DataFrame({
    "Métrica": ["R2", "RMSE", "MAE"],
    "Valor": [r2, rmse, mae]
})
metricas

In [ ]:
coeficientes = pd.DataFrame({
    "Variable": features,
    "Coeficiente": modelo.coef_
})
coeficientes

## 6. Conclusiones

- La categoría Music domina en cantidad de videos, visualizaciones y likes.
- Los canales más populares concentran una gran proporción de las interacciones.
- Existe una relación positiva entre likes y views, aunque no todos los videos se comportan igual.
- La regresión lineal permite una primera aproximación al modelado, pero se recomienda comparar con modelos adicionales en trabajos futuros.